In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [43]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 8, 9, tzinfo=timezone)    
#     utc_to = datetime(x.year, x.month, x.day, tzinfo=timezone)
#     utc_to = datetime(x.year, x.month+1, 9, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M1, utc_from, x)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
#     rates_frame['sma'] = rates_frame['close'].rolling(window=100).mean()
#     rates_frame['smaH'] = rates_frame['high'].rolling(window=20).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    return rates_frame

In [44]:
def get_rsi(close, lookback):
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     rsi_df = rsi_df.dropna()
    return rsi_df

# ibm['rsi_14'] = get_rsi(ibm['close'], 14)
# ibm = ibm.dropna()


In [45]:
def slope(x1, y1, x2, y2):
    return (y2-y1)/(x2-x1)

In [46]:
def go(a):
    g = []
    for i in range(0, len(a)):
        if a.iloc[i].rsi >= 60.0:
            g.append(slope(0, a.iloc[i-5].rsi, 5, a.iloc[i].rsi))
        else:
            g.append(0)
    return g

In [62]:
symbol = "USDCHF"
a= get_values(symbol)
a['rsi'] = get_rsi(a['close'], 30)
a['smaL']= a['rsi'].rolling(window=2).mean()
a['slope'] = go(a)
a = a.dropna()
a = a[100:]
a

,open,close,rsi,smaL,slope
time,,,,,
2021-08-09 01:47:00,0.91549,0.91551,57.870580,57.629195,0.000000
2021-08-09 01:48:00,0.91551,0.91553,58.358620,58.114600,0.000000
2021-08-09 01:49:00,0.91553,0.91554,58.606644,58.482632,0.000000
2021-08-09 01:50:00,0.91558,0.91547,56.183396,57.395020,0.000000
2021-08-09 01:51:00,0.91547,0.91579,63.349817,59.766606,1.192401
...,...,...,...,...,...
2021-08-09 10:24:00,0.91560,0.91526,48.122272,50.981044,0.000000
2021-08-09 10:25:00,0.91526,0.91543,50.823474,49.472873,0.000000
2021-08-09 10:26:00,0.91543,0.91536,49.720702,50.272088,0.000000


In [63]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if a.iloc[i].rsi >= 41.0:
        if a.iloc[i].slope > 1.9 and a.iloc[i].slope > 0.0 and a.iloc[i-1].slope == 0.0 \
            and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("RSI   SLOPE")
            print(f"{round(a.iloc[i].rsi,2)}   {round(a.iloc[i].slope,2)}")
#             print("SMA   SMAH")
#             print(f"{peck}   {up}")
# #             print(f"{round(a.iloc[i].sma,6)}   {round(a.iloc[i].smaH,6)}")
#             print("CLOSE")
#             print(f"{a.iloc[i].Close}")
#             print(a.iloc[i].smaL)
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(f"{pp}---{round(a.iloc[i].rsi, 2)}--{round(a.iloc[i].slope, 2)}---{a.iloc[i].close}--{a.iloc[i].name}")
#             profit.append(pp)
#             check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
            if pp < -3.0:
                profit.append(pp)
                check = 0
#             if a.iloc[i].rsi >= 49.0:
#                 if a.iloc[i].smaL >= a.iloc[i-1].smaL:
#                     pass
#                 else:
#                     profit.append(pp)
#                     check = 0
#             if a.iloc[i].rsi - a.iloc[i-1].rsi >= 2.0 :
#                 profit.append(pp)
#                 check = 0
            if a.iloc[i].rsi <= 41.0:
                profit.append(pp)
                check = 0
#             if a.iloc[i].rsi <= 40.0:
#                 profit.append(pp)
#                 check = 0

####################
2021-08-09 09:00:00
RSI   SLOPE
62.23   2.39
********************
-0.13---64.89--2.76---0.91509--2021-08-09 09:01:00
0.24---53.78--0.0---0.91492--2021-08-09 09:02:00
0.42---49.65--0.0---0.91484--2021-08-09 09:03:00
0.37---50.63--0.0---0.91486--2021-08-09 09:04:00
0.61---45.57--0.0---0.91475--2021-08-09 09:05:00
0.42---49.81--0.0---0.91484--2021-08-09 09:06:00
0.39---50.26--0.0---0.91485--2021-08-09 09:07:00
0.63---45.65--0.0---0.91474--2021-08-09 09:08:00
0.79---43.05--0.0---0.91467--2021-08-09 09:09:00
0.92---40.98--0.0---0.91461--2021-08-09 09:10:00


In [59]:
sum(profit)

0

In [60]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

Total negative sm -->0
Total negative -->0
Total positive sm -->0
Total positive -->0
Length 0


In [ ]:
counterr = 0
peck = 0
for i in profit:
    if i < 0.0:
        counterr = counterr + i
    else:
        peck = peck + i
print(counterr)
print(peck)

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if str(a.iloc[i].rsi) != 'nan':
        if a.iloc[i].rsi >= 50.0 and a.iloc[i-1].rsi >= 50.0 and a.iloc[i-2].rsi <= 50.0 \
            and a.iloc[i-3].rsi <= 50.0 and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print(round(a.iloc[i].rsi,2))
            print("*"*20)
            check = 1 
            boght = a.iloc[i].rsi

        elif check == 1:
            sell_price = a.iloc[i].close
            g = round(a.iloc[i].rsi - boght, 2)
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(f"{pp}---{round(a.iloc[i].rsi, 2)}---{a.iloc[i].name}   {g}")
#             profit.append(pp)
#             check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
            if pp < -10.0:
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi >= 60.0:
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi - a.iloc[i-1].rsi <= -2.0 :
                profit.append(pp)
                check = 0
                print("HERE")
            if a.iloc[i].rsi < 50.0:
                profit.append(pp)
                check = 0

In [ ]:
# a['rsi'].plot(figsize=(15,6))
# rates_frame['ll'] = rates_frame['close'].rolling(window=100).mean()
a['slope'].plot(figsize=(15,6))

In [ ]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if str(a.iloc[i].rsi) != 'nan':
        if a.iloc[i].rsi >= 50.0 and a.iloc[i-1].rsi >= 50.0 and a.iloc[i-2].rsi <= 50.0 \
            and a.iloc[i-3].rsi <= 50.0 and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print(round(a.iloc[i].rsi,2))
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---", round(a.iloc[i].rsi, 2), "---", a.iloc[i].name)
#             profit.append(pp)
#             check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
            if pp < -10.0:
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi >= 60.0:
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi - a.iloc[i-1].rsi >= 2.0 :
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi >= 60.0:
                profit.append(pp)
                check = 0

In [ ]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if str(a.iloc[i].rsi) != 'nan':
        if a.iloc[i].rsi >= 50.0 and a.iloc[i-1].rsi <= 50.0 and a.iloc[i-2].rsi <= 50.0 \
             and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print(round(a.iloc[i].rsi,2))
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---", round(a.iloc[i].rsi, 2), "---", a.iloc[i].name)
#             profit.append(pp)
#             check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
            if pp < -10.0:
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi <= 49.0:
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi - a.iloc[i-1].rsi >= 2.0 :
                profit.append(pp)
                check = 0
            if a.iloc[i].rsi >= 60.0:
                profit.append(pp)
                check = 0

In [ ]:
#ALL Positive results for 1minute chart rsi at 30 and rsi avg at 2

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
    if a.iloc[i].rsi >= 41.0:
        if a.iloc[i].smaL >=42.0 and a.iloc[i-1].smaL <= 40.0 and a.iloc[i-1].smaL <= 40.0\
             and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print(round(a.iloc[i].rsi,2))
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 1.0, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---", round(a.iloc[i].rsi, 2), "---", a.iloc[i].name)
            profit.append(pp)
            check = 0
            
            
#             if pp > 0.0:
#                 profit.append(pp)
#                 check = 0
#             if pp < -10.0:
#                 profit.append(pp)
#                 check = 0
#             if a.iloc[i].rsi <= 49.0:
#                 profit.append(pp)
#                 check = 0
#             if a.iloc[i].rsi - a.iloc[i-1].rsi >= 2.0 :
#                 profit.append(pp)
#                 check = 0
#             if a.iloc[i].rsi >= 60.0:
#                 profit.append(pp)
#                 check = 0
#             if a.iloc[i].rsi < 50.0:
#                 profit.append(pp)
#                 check = 0